# P5-1 · GĐ 1 · Diffusion tạo ảnh thế nào

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, math
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)     # tích lũy: ᾱ₁, ᾱ₂, ..., ᾱ₁₀₀₀

def q_sample(x0, t, eps):
    ab = alpha_bars[t].unsqueeze(1)            # (B,) -> (B,1) để nhân với x0 có shape (B,2)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

In [ ]:
import torch.nn as nn

def time_emb(t, dim=32):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)
    ang = t.float().unsqueeze(1) * freqs.unsqueeze(0)          # (B, half)
    return torch.cat([ang.sin(), ang.cos()], dim=1)             # (B, dim)

class Denoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 32, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, time_emb(t)], dim=1))     # ghép xₜ với embedding của t

s = torch.rand(4000).sqrt() * 3 * math.pi                       # 4000 điểm trên xoắn ốc
data = torch.stack([s * s.cos(), s * s.sin()], 1) / 5

model = Denoiser(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(3000):
    x0 = data[torch.randint(0, len(data), (256,))]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    loss = ((model(q_sample(x0, t, eps), t) - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

In [ ]:
@torch.no_grad()
def sample(model, n=400):
    x = torch.randn(n, 2)                                    # x₁₀₀₀: nhiễu thuần
    for t in reversed(range(T)):                             # 999, 998, ..., 0
        eps_hat = model(x, torch.full((n,), t))
        x = (x - betas[t] / (1 - alpha_bars[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)    # nhiễu mới, trừ bước cuối
    return x